# 네이버 증권 뉴스 탭의 3일치 기사를 수집
데이터 프레임으로 생성하고 파일롤 저장
사이트 주소: https://finance.naver.com/news/mainnews.naver?date=2024-10-25

In [10]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import re

# ── 기본 설정 ──────────────────────────────────────────
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"})

BASE_URL = "https://finance.naver.com/news/mainnews.naver?date={}"

# 오늘 기준 3일치 날짜 자동 생성
today     = datetime.today()
date_list = [(today - timedelta(days=i)).strftime('%Y-%m-%d') for i in range(3)]
print("수집 날짜:", date_list)

# ── redirect URL 추출 함수 ─────────────────────────────
def resolve_redirect(naver_url):
    try:
        res        = session.get(naver_url, timeout=10)
        script_tag = BeautifulSoup(res.text, 'lxml').find('script')
        if script_tag and "'" in script_tag.text:
            return script_tag.text.split("'")[1]
    except Exception:
        pass
    return None

# ── 본문 수집 함수 ─────────────────────────────────────
def fetch_article_body(real_url):
    try:
        res      = session.get(real_url, timeout=10)
        res.raise_for_status()
        news_soup = BeautifulSoup(res.text, 'lxml')
        for selector in ['div#articeBody', 'div#articleBodyContents',
                         'div.article_body', 'div#newsct_article']:
            body = news_soup.select_one(selector)
            if body:
                return re.sub(r'\s+', ' ', body.text.strip())
        return "본문 없음"
    except Exception:
        return "본문 없음"

# ── 3일치 기사 수집 ────────────────────────────────────
raw_articles = []

for date in date_list:
    response = session.get(BASE_URL.format(date), timeout=10)
    soup     = BeautifulSoup(response.text, 'lxml')
    titles   = soup.find_all('dd', {'class': 'articleSubject'})
    print(f"\n[{date}] 기사 수: {len(titles)}개")

    for item in titles:
        a_tag = item.find('a')
        if not a_tag:
            continue

        title_text = a_tag.text.strip()
        naver_url  = 'https://finance.naver.com' + a_tag.get('href', '')
        real_url   = resolve_redirect(naver_url)
        content    = fetch_article_body(real_url) if real_url else "본문 없음"

        raw_articles.append({
            'date'    : date,
            'title'   : title_text,
            'real_url': real_url or "redirect 실패",
            'content' : content
        })
        print(f"  ✔ {title_text[:35]}...")

# ── DataFrame 생성 + 클린징 ────────────────────────────
news_df = pd.DataFrame(raw_articles)
news_df.insert(0, 'rank', range(1, len(news_df) + 1))
news_df['is_missing'] = news_df['content'] == "본문 없음"

# ── 시각적으로 정리된 표 출력 ──────────────────────────
pd.set_option('display.max_rows', 30)
pd.set_option('display.width', 130)
pd.set_option('display.max_colwidth', 35)
pd.set_option('display.colheader_justify', 'center')

display_df = news_df[['rank', 'date', 'title', 'is_missing']].copy()
display_df.columns = ['순위', '날짜', '제목', '본문없음']
display_df = display_df.set_index('순위')

print("\n\n[ 네이버 증권 뉴스 수집 결과 ]")
print(display_df.to_string())

# ── 수집 통계 요약 (표와 분리) ─────────────────────────
missing_count = news_df['is_missing'].sum()
total_count   = len(news_df)

print("\n" + "=" * 50)
print("[ 수집 통계 요약 ]")
print("=" * 50)
print(f"  수집 날짜  : {', '.join(date_list)}")
print(f"  전체 기사  : {total_count}개")
print(f"  본문 없음  : {missing_count}개")
print(f"  수집 성공률: {(total_count - missing_count) / total_count:.1%}")
print("=" * 50)

# ── CSV 저장 ───────────────────────────────────────────
news_df.to_csv("naver_finance_news.csv", encoding="utf-8-sig", index=False)
print("\nSave complete → naver_finance_news.csv")

수집 날짜: ['2026-02-25', '2026-02-24', '2026-02-23']

[2026-02-25] 기사 수: 20개
  ✔ 6000 축포 쏜 코스피… 과열 경고등도 함께 점화...
  ✔ 코스피 6000 돌파 새 역사…“이젠 7000피 간다”...
  ✔ 코스피 랠리 주도한 개미...강력한 머니무브 주역으로 올라서...
  ✔ 관리종목 해제 가능할까…인스코비, ‘상폐 분수령’ 결산 감사 촉...
  ✔ '3차 상법개정안' 통과…"금융주 투자 매력 여전"...
  ✔ 한달여만 1,000포인트 올랐다…압도적 세계 1위(종합)[코스피...
  ✔ 이제는 '코스피 6000' 시대…시가총액도 5000조 돌파...
  ✔ 소각 의무법 지연 틈 타…자사주 처분 막차타기 '눈총'...
  ✔ '육천피' 불장에 공매도 대기자금 '역대 최대'…대차잔고153조...
  ✔ '육천피' 기관 투자자가 만들었다...개미 8조 던질 때 기관이...
  ✔ 40만전자·200만닉스도 멀지 않았다…8000피 시대 성큼...
  ✔ '신고가 경신' 삼전·하이닉스 영업익 연 300兆 '기대'...
  ✔ 코스피 강세에…외국인 코스피 시가총액 1800조 돌파...
  ✔ 57만원대 9%↑ 최고가 경신…'6천피시대' 현대차 강세 [신G...
  ✔ '자사주 소각' 기업 벌써 2배 늘었다..."주가 들썩" 상법개...
  ✔ '6000' 넘은 코스피, 어디까지 갈까… '개인·외국인' 동향...
  ✔ MBK, 홈플러스 1000억 선투입 ‘배수진’…메리츠 ‘묵묵부답...
  ✔ 코스피 마침내 6000 고지…시총 5000조 열렸다...
  ✔ 코스피 PBR 2배 시대…증권가 꼽은 저평가 업종은...
  ✔ 고공행진 코스피, '오천피' 한 달 만에 6000마저 넘었다...

[2026-02-24] 기사 수: 20개
  ✔ 20만전자·100만닉스 찍더니…'깜짝 전망' 내놓은 증권가 [종...
  ✔ 애프터마켓서도 '20만전자', '백만닉스' 강세 계속...
  ✔ “은퇴자금까지 탈탈 털어 넣

# 네이버 증권 뉴스 탭의 3일치 기사를 수집
데이터 프레임으로 생성하고 파일롤 저장
사이트 주소: https://finance.naver.com/news/mainnews.naver?date=2024-10-25

In [9]:
# 날짜별 제목 + href 동시 추출 (원본은 제목만 추출)
raw_articles = []

for date, soup in soup.items():
    titles = soup.find_all('dd', {'class': 'articleSubject'})
    print(f"[{date}] 기사 수: {len(titles)}개")

    for item in titles:
        a_tag = item.find('a')
        if a_tag:
            title_text = a_tag.text.strip()
            href       = 'https://finance.naver.com' + a_tag.get('href', '')
            raw_articles.append({'date': date, 'title': title_text, 'naver_url': href})

print(f"\n전체 수집 기사 수: {len(raw_articles)}개")

# 첫 번째 기사 미리보기
print("\n[첫 번째 기사 확인]")
print(f"  제목 : {raw_articles[0]['title']}")
print(f"  URL  : {raw_articles[0]['naver_url']}")

TypeError: 'NoneType' object is not callable

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

url = 'https://finance.naver.com/news/mainnews.naver?date=2026-02-20'
response = requests.get(url)
soup = BeautifulSoup(response.text, 'lxml')

In [2]:
# 기사제목 리스트 가져오기
titles = soup.find_all('dd', {'class':'articleSubject'})
print(titles[:2])

# 첫번째 기사의 기사 제목 
print(titles[0].text.strip())

[<dd class="articleSubject">
<a href="/news/news_read.naver?article_id=0001242470&amp;office_id=215&amp;mode=mainnews&amp;type=&amp;date=2026-02-20&amp;page=1">2월 10일 옵션 만기의 폭풍 속에서도 기관의 강력한 드라이브가 포착... &lt;진짜주식 1부 : 시장의 알리바이&gt;</a>
</dd>, <dd class="articleSubject">
<a href="/news/news_read.naver?article_id=0001242469&amp;office_id=215&amp;mode=mainnews&amp;type=&amp;date=2026-02-20&amp;page=1">2차전지·남북 경협·반도체 등 전문가 심층 분석, MSCI 선진 지수 편입 등 시장 이벤트 집중 조명… &lt;진짜주식 1부 : 시장의 알리바이&gt;</a>
</dd>]
2월 10일 옵션 만기의 폭풍 속에서도 기관의 강력한 드라이브가 포착... <진짜주식 1부 : 시장의 알리바이>


In [3]:
# 첫번째 기사의 상세페이지 URL

titles[0].find('a').get('href')
news_url = 'https://finance.naver.com' + titles[0].find('a').get('href')

# 상세 뉴스페이지 가져오기
news_response = requests.get(news_url)
news_soup = BeautifulSoup(news_response.text, 'lxml')
print(news_soup)

<html><head><script>top.location.href='https://n.news.naver.com/mnews/article/215/0001242470';</script>
</head></html>


In [4]:
#redirect 주소를 반환하고 있으므로, redirecet 주소를 사용하여 다시 상세 뉴스 페이지에 접근
news_url2 = news_soup.find('script').text.split("'")[1]

# 상세 뉴스페이지 재도전
news_response2 = requests.get(news_url2)
news_soup = BeautifulSoup(news_response2.text, 'lxml')
print(news_soup.prettify()[:1000])

<!DOCTYPE html>
<html data-useragent="python-requests/2.32.5" lang="ko">
 <head>
  <meta charset="utf-8"/>
  <meta content="IE=edge" http-equiv="X-UA-Compatible"/>
  <meta content="width=device-width, initial-scale=1.0, maximum-scale=1.0, minimum-scale=1.0, user-scalable=no" name="viewport"/>
  <meta content="2월 12일 옵션 만기의 폭풍 속에서도 기관의 강력한 드라이브가 포착... &lt;진짜주식 1부 : 시장의 알리바이&gt;" property="og:title"/>
  <meta content="article" property="og:type"/>
  <meta content="https://n.news.naver.com/mnews/article/215/0001242470" property="og:url"/>
  <meta content="https://imgnews.pstatic.net/image/215/2026/02/20/A202602130281_1_20260222101210203.png?type=w800" property="og:image"/>
  <meta content="2월 12일 옵션 만기의 폭풍 속에서도 기관의 강력한 드라이브가 포착됐다. 삼성전자는 17만원 고지를 점령하며 20만 전자라는 새로운 도전에 나섰고, 코스피는 5500 축포를 쏘아 올렸다. 이 화려한 축제의 환호 속" property="og:description"/>
  <meta content="한국경제TV | 네이버" property="og:article:author"/>
  <meta content="summary_large_image" name="twitter:card"/>
  <meta content="2월 12일 옵션 만기의 폭

# 상세 뉴스 페이지 본문 파싱하기

In [5]:
# 뉴스 본문
# 기사 본문: <div id="newsct_article" class="newsct_article _article_body">
news_1 = news_soup.find('div', {'class':'newsct_article _article_body'}).text.strip()
print(news_1)

삼성전자 상승세, 20만원 도전 전망방산·반도체·유통 섹터별 시장 분석외국인 관광객 증가, 유통·백화점주 호재2월 12일 옵션 만기의 폭풍 속에서도 기관의 강력한 드라이브가 포착됐다. 삼성전자는 17만원 고지를 점령하며 20만 전자라는 새로운 도전에 나섰고, 코스피는 5500 축포를 쏘아 올렸다. 이 화려한 축제의 환호 속에서 우리가 봐야 할 진짜 트리거는 무엇일지, 시장에서 벌어진 사건의 결정적인 알리바이를 파헤쳐 하나씩 추적하는 주식 추리 토크쇼 <진짜주식 1부 : 시장의 알리바이>가 2월 12일 시청자들을 찾았다.이날 방송에는 시장의 사건을 함께 추적할 세 명의 전문가가 출연했다.- 임주아(시그널 탐정): 시장의 수급과 투자 심리를 포착하는 전문가- 전태룡(수사반장): 사건의 과거와 구조 속에서 같은 장면을 추적하는 전문가- 홍의진(프로파일러): 섹터와 판의 변화 속에서 사건들을 하나의 흐름으로 정리하는 전문가



■ 임주아 대표의 시장 분석: 방산 BIG 3 폭주, 숨겨진 진범은 시스템임주아 대표는 1월부터 2월 첫째 주까지 ETF의 섹터별 대장주들이 연간 목표 주가까지 도달해 신규 진입이 부담스러운 상황이라고 진단했다. 개인 물량을 털어내고 재매수하는 단기적인 흐름 외에는 특이 사항이 없었으며, 설 연휴 이후 섹터 변화를 주시하거나 흘러내린 종목의 반등을 노리는 트레이딩이 유효할 것으로 예상했다.방산 섹터에 대해서는 ▲한화에어로스페이스(012450)가 이미 목표 주가가 수주 물량만큼 반영돼 개인 투자자의 단독 트레이딩이 어렵다고 분석했다. ▲한화오션(042660) 역시 방산 밸류가 포함된 주가로 오버슈팅이 발생하지 않고 있다고 설명했다. 임주아 대표는 기계 산업군(조선, 방산, 원전, 로봇, 항공) 내에서 가장 덜 오른 섹터로 수급이 순환하고 있다고 덧붙였다.특히 ▲한화시스템(272210)의 상승 원인으로 K2 전차 수주나 필리조선소 내용이 아닌, 드론 요격 레이저 빔 장비인 '천광'의 유럽 전시 반응이 좋았기 때문이라고 지적했다. ▲한화시스템(27

# 함수로 만들기

In [6]:
#반복문 사용하여 함수로 생성하기
import datetime
import time

def get_news_items(html):
  # 기사제목 리스트 가져오기
  titles = html.find_all('dd', {'class':'articleSubject'})
  title_list = []
  url_list = []
  article_list = []

  for t in titles:
    # 기사의 제목
    title = t.text.strip()

    # 기사의 상세페이지 URL
    news_url = 'https://finance.naver.com' + t.find('a').get('href')

    # 상세페이지 request
    news_response = requests.get(news_url)
    news_soup = BeautifulSoup(news_response.text, 'lxml')

    # 리다이렉트 주소를 파싱
    news_url2 = news_soup.find('script').text.split("'")[1]

    # 상세 뉴스페이지 request 재도전
    news_response2 = requests.get(news_url2)
    news_soup = BeautifulSoup(news_response2.text, 'lxml')

    # 신문기사 본문 파싱
    article = news_soup.find('div', {'class':'newsct_article _article_body'}).text.strip()

    # 리스트에 값 채우기
    title_list.append(title)
    url_list.append(news_url2)
    article_list.append(article)

  df = pd.DataFrame({'기사제목': title_list, '본문url': url_list, '기사본문': article_list})
  return df

In [7]:
news_df = pd.DataFrame()

for i in range(3):
    # 오늘 날짜 기준 i일 이전 날짜 구하기
    date= datetime.date.today() - datetime.timedelta(days = i)
    url = f'https://finance.naver.com/news/mainnews.naver?date={date}'
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'lxml')
    time.sleep(1)

    # 해당일 신문기사 스크랩 데이터프레임 반환
    temp_df = get_news_items(soup)
    temp_df['date'] = date
    news_df = pd.concat([news_df, temp_df], ignore_index=True).copy()

    print(news_df.head())
    news_df.to_csv('article.csv')

                                           기사제목  \
0  "추론 AI 시대, 메모리 병목 심화"…맥쿼리, 삼성전자 주가 '34만원' 제시   
1       이승훈 IBK증권 센터장 "코스피 추가 상승 여력 있어"[육천피 시대]   
2           비트코인 9200만원대 지지 후 반등…뉴욕증시 훈풍에 숨 고르기   
3    한달여만 1,000포인트 올랐다…올해도 압도적 세계 1위[코스피 6,000]   
4   20만전자 끌고, 100만닉스 밀어 '코스피 8000'간다[코스피6000돌파]   

                                               본문url  \
0  https://n.news.naver.com/mnews/article/215/000...   
1  https://n.news.naver.com/mnews/article/003/001...   
2  https://n.news.naver.com/mnews/article/243/000...   
3  https://n.news.naver.com/mnews/article/001/001...   
4  https://n.news.naver.com/mnews/article/277/000...   

                                                기사본문        date  
0  메모리 반도체 업황을 바라보는 외국계 증권사의 시선이 한층 공격적으로 변했다. 맥쿼...  2026-02-25  
1  "이르면 2분기 코스피 7000선 돌파 예상"\n\n\n\n이승훈 IBK투자증권 리...  2026-02-25  
2  美 3대 지수 일제히 상승 마감\n\n\n\n 비트코인 [사진 연합뉴스][이코노미스...  2026-02-25  
3  상승 속도도 최고…3,000→4,000 넉달·4,000→5,000 석달·5,000→...  2026-02-25  
4  삼전 30만, SK하닉 160만 